# Create a Versioned Foundry Agent

This notebook creates a **versioned agent** on Microsoft Foundry using the
[`azure-ai-projects`](https://pypi.org/project/azure-ai-projects/) SDK.

**What this notebook does:**
1. Authenticates to Azure using `DefaultAzureCredential`
2. Creates or increments the version of a named agent using `create_version`
3. Sends a message through the OpenAI-compatible responses API and prints the reply

> **`create_version` is idempotent** — re-running bumps the version only when the agent
> definition changes, making it safe to iterate on prompts and model settings during development.

## Prerequisites

1. **Python environment**: Run `uv sync` from the repository root to create the
   shared `.venv`, then select the `.venv` kernel in VS Code.
2. **`.env` file**: Must be populated by the `04-foundry-project-pattern-setup` labs:
   - `ALPHA_FOUNDRY_PROJECT_ENDPOINT` — Team Alpha project endpoint URL (set by Lab 1B)
   - `ALPHA_FOUNDRY_CORE_CONNECTION` — Team Alpha APIM connection name, e.g. `core-alpha` (set by Lab 1B)
   - `CHAT_MODEL` — chat model deployment name, e.g. `gpt-4.1-mini` (set by Lab 1A)
3. **Azure CLI**: Run `az login` before executing the cells.
4. **Permissions**: Your identity needs **Azure AI Developer** role on the Foundry project.

## 1. Imports and configuration

Load `.env` from the repository root and read the required environment variables.

In [1]:
import os
import subprocess
from pathlib import Path
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

# ── Change this to rename your agent ─────────────────────────────────────────
AGENT_NAME = "storytelling-agent"
# ─────────────────────────────────────────────────────────────────────────────

repo_root = Path(subprocess.run(
    'git rev-parse --show-toplevel', shell=True, capture_output=True, text=True
).stdout.strip())
load_dotenv(repo_root / '.env', override=True)

# Team Alpha (1:1 spoke) — project endpoint and APIM connection (set by Labs 1A / 1B)
endpoint       = os.environ["ALPHA_FOUNDRY_PROJECT_ENDPOINT"]
hub_connection = os.environ["ALPHA_FOUNDRY_CORE_CONNECTION"]  # e.g. "core-alpha"
chat_model     = os.environ["CHAT_MODEL"]                    # e.g. "gpt-4.1-mini"

# Agents reference models as {connection}/{model} — routes through the APIM connection
model_deployment = f"{hub_connection}/{chat_model}"

print(f"Endpoint  : {endpoint}")
print(f"Agent name: {AGENT_NAME}")
print(f"Model     : {model_deployment}")

Endpoint  : https://aif-spoke-alpha-c2676f.services.ai.azure.com/api/projects/project-alpha-c2676f
Agent name: storytelling-agent
Model     : core-alpha/gpt-4.1-mini


## 3. Configure authentication

`DefaultAzureCredential` resolves credentials automatically using the az CLI login, VS Code
sign-in, managed identity, or environment variables — no manual token management required.

In [2]:
credential = DefaultAzureCredential()

## 4. Create the project client

`AIProjectClient` is the main entry point for the Foundry Agent Service. It provides access
to agent management and model inference operations via the project endpoint.

In [3]:
project_client = AIProjectClient(
    endpoint=endpoint,
    credential=credential,
)

## 5. Create or version the agent

`create_version` creates the agent on first run. On subsequent runs it compares the
`PromptAgentDefinition` to the latest stored version and only creates a new version when
something has changed — safe to re-run while iterating on instructions or model settings.

- **`model`** — references a model deployment configured in your Foundry project
- **`instructions`** — the system-level prompt that shapes the agent's behaviour

In [4]:
agent = project_client.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=model_deployment,
        instructions=(
            "You are a storytelling agent. "
            "You craft engaging one-line stories based on user prompts and context."
        ),
    ),
)

print(f"Agent name: {agent.name}")

Agent name: storytelling-agent


## 6. Send a message to the agent

Use the OpenAI-compatible responses API to send a message. Setting `type` to
`agent_reference` routes the request through the named agent, applying its stored
instructions and model configuration automatically.

In [5]:
openai_client = project_client.get_openai_client()

response = openai_client.responses.create(
    input=[{"role": "user", "content": "Tell me a one line story"}],
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
)

print(f"Response: {response.output_text}")

Response: Beneath the silent stars, she whispered forgotten dreams into the night, awakening a world long asleep.
